In [32]:
# This script is modified to use the IGN/CDDIS public FTP mirror for navigation data.

import os
from datetime import datetime
import numpy as np
import requests
import gzip
from glob import glob
import sys
from math import radians, cos, sin, asin, sqrt

# Input parameters
datadir = r'C:\Users\gauth\Documents\SC4000_proj\train' 
stas = ['slac', 'vdcy', 'p222']
obs_url_base = 'https://geodesy.noaa.gov/corsdata/rinex'
nav_url_base = 'https://igs.bkg.bund.de/root_ftp/IGS/BRDC' 
crx2rnx_bin = r"C:\Users\gauth\Documents\RTKLIB\crx2rnx.exe"

# Base station coordinates (lat, lon)
STATIONS = {
    "slac": (37.417, -122.204),
    "vdcy": (34.179, -118.220),
    "p222": (37.539, -122.083),
}

# Known dataset locations
LOCATIONS = {
    "MTV": (37.4046, -122.0764),
    "SVL": (37.3688, -121.9269),
    "SJC": (37.3382, -121.8863),
    "LAX": (33.9416, -118.4085),
    "SFO": (37.6213, -122.3790),
    "OAK": (37.7214, -122.2208),
}

def haversine_distance(lat1, lon1, lat2, lon2):
    """Calculate distance in km between two lat/lon points"""
    lon1, lat1, lon2, lat2 = map(radians, [lon1, lat1, lon2, lat2])
    dlon = lon2 - lon1
    dlat = lat2 - lat1
    a = sin(dlat/2)**2 + cos(lat1) * cos(lat2) * sin(dlon/2)**2
    c = 2 * asin(sqrt(a))
    km = 6371 * c
    return km

def get_location_from_dataset(dataset_name):
    """Extract location code from dataset folder name"""
    parts = dataset_name.split('-')
    if len(parts) >= 5:
        return parts[4]  # e.g., "2020-05-15-US-MTV-1" -> "MTV"
    return None

def get_closest_stations(location_code, available_stations):
    """Return list of stations sorted by distance to location"""
    if location_code not in LOCATIONS:
        print(f"Warning: Unknown location {location_code}, using default order")
        return available_stations
    
    loc_lat, loc_lon = LOCATIONS[location_code]
    
    # Calculate distances to all available stations
    distances = []
    for sta in available_stations:
        if sta in STATIONS:
            sta_lat, sta_lon = STATIONS[sta]
            dist = haversine_distance(loc_lat, loc_lon, sta_lat, sta_lon)
            distances.append((dist, sta))
        else:
            distances.append((999, sta))  # Unknown station, put at end
    
    # Sort by distance
    distances.sort()
    sorted_stations = [sta for dist, sta in distances]
    
    print(f"Station order for {location_code}: {[f'{sta}({dist:.1f}km)' for dist, sta in distances[:3]]}")
    return sorted_stations

os.chdir(datadir)

for dataset in np.sort(os.listdir()):
    if not os.path.isdir(dataset):
        continue

    print("Processing:", dataset)
    ymd = dataset.split('-')
    
    # Check for valid date parts before processing
    if len(ymd) < 3:
        print(f"Skipping {dataset}: Invalid date format.")
        continue
        
    try:
        year = ymd[0]
        month = ymd[1]
        day = ymd[2]
        
        # Calculate DOY and get short year
        dt_obj = datetime(int(year), int(month), int(day))
        doy = dt_obj.timetuple().tm_yday
        doy_str = str(doy).zfill(3)
        year_short = year[2:4]
    except ValueError:
        print(f"Skipping {dataset}: Invalid date value.")
        continue
    
    # Get location and sort stations by distance
    location = get_location_from_dataset(dataset)
    sorted_stations = get_closest_stations(location, stas)
    
    # === BEGIN: BASE STATION OBSERVATION DATA BLOCK ===
    # Delete old observation files to force re-download with new closest station
    old_obs_files = glob(os.path.join(dataset,'*.*o')) + glob(os.path.join(dataset,'*.*d'))
    for old_file in old_obs_files:
        try:
            os.remove(old_file)
            print(f"Deleted old file: {os.path.basename(old_file)}")
        except Exception as e:
            print(f"Could not delete {old_file}: {e}")
    
    success = False
    
    # Try stations in order of distance (closest first)
    for idx, station_id in enumerate(sorted_stations):
        # Filename example: slac1360.20d.gz 
        fname_base = station_id + doy_str + '0.' + year_short + 'd.gz'
        
        # Path structure: /YYYY/DOY/station/filename
        obs_url = '/'.join([obs_url_base, year, doy_str, station_id, fname_base])
        
        # *** DEBUG LINE ***
        print(f"Attempting BASE OBS from {station_id} (attempt {idx+1}/{len(sorted_stations)}): {obs_url}")
        # ******************
        
        try:
            # Check if file exists and download/decompress
            response = requests.get(obs_url, timeout=10)
            response.raise_for_status() # Raise exception for 4xx or 5xx status codes
            
            obs = gzip.decompress(response.content) # get obs and decompress
            
            # Decompressed filename: slac1360.20d
            final_obs_filename = fname_base[:-3]
            
            # write obs data
            open(os.path.join(dataset, final_obs_filename), "wb").write(obs)
            print(f"✓ Success: Base OBS file written to {final_obs_filename} using {station_id}")
            success = True
            break  # Exit loop on success

        except Exception as e:
            print(f"✗ Failed {station_id}: {e}")
            continue  # Try next station
    
    if not success:
        print(f"✗✗ FAILED ALL STATIONS for {dataset}")

    # convert crx to rnx (This section runs regardless of failure above)
    crx_files = glob(os.path.join(dataset,'*.*d'))
    if len(crx_files) > 0:
        # Note: This external command execution may fail silently if the executable path is wrong
        # or if the file conversion fails.
        os.system(crx2rnx_bin + ' ' + crx_files[0])
    # === END: BASE STATION OBSERVATION DATA BLOCK ===
    
    
    # === BEGIN: NAVIGATION DATA BLOCK ===
    # Delete old navigation files to force re-download
    old_nav_files = glob(os.path.join(dataset,'*.rnx'))
    for old_file in old_nav_files:
        try:
            os.remove(old_file)
            print(f"Deleted old nav file: {os.path.basename(old_file)}")
        except Exception as e:
            print(f"Could not delete {old_file}: {e}")
    
    # Filename format: BRDC00WRD_R_YYYYDOY0000_01D_MN.rnx.gz
    fname = f'BRDC00WRD_R_{year}{doy_str}0000_01D_MN.rnx.gz'
    
    # Path structure: /daily/YYYY/DOY/filename
    url = '/'.join([nav_url_base, year, doy_str, fname])
    
    # *** DEBUG LINE ***
    print(f"Attempting NAV file download from URL: {url}")
    # ******************
    
    try:
        response = requests.get(url, timeout=10)
        response.raise_for_status() 

        obs = gzip.decompress(response.content) 
        
        final_filename = fname[:-3]
        
        open(os.path.join(dataset, final_filename), "wb").write(obs)
        print(f"✓ NAV file downloaded")
        
    except requests.exceptions.RequestException as e:
        print(f'✗ Fail nav (Request Error): {dataset} - {e}')
        
    except Exception as e:
        print(f'✗ Fail nav (General Error): {dataset} - {e}')

Processing: 2020-05-15-US-MTV-1
Station order for MTV: ['slac(11.4km)', 'p222(15.0km)', 'vdcy(499.5km)']
Deleted old file: slac1360.20o
Deleted old file: slac1360.20d
Attempting BASE OBS from slac (attempt 1/3): https://geodesy.noaa.gov/corsdata/rinex/2020/136/slac/slac1360.20d.gz
✓ Success: Base OBS file written to slac1360.20d using slac
Deleted old nav file: BRDC00WRD_R_20201360000_01D_MN.rnx
Attempting NAV file download from URL: https://igs.bkg.bund.de/root_ftp/IGS/BRDC/2020/136/BRDC00WRD_R_20201360000_01D_MN.rnx.gz
✓ NAV file downloaded
Processing: 2020-05-21-US-MTV-1
Station order for MTV: ['slac(11.4km)', 'p222(15.0km)', 'vdcy(499.5km)']
Deleted old file: slac1420.20o
Deleted old file: slac1420.20d
Attempting BASE OBS from slac (attempt 1/3): https://geodesy.noaa.gov/corsdata/rinex/2020/142/slac/slac1420.20d.gz
✓ Success: Base OBS file written to slac1420.20d using slac
Deleted old nav file: BRDC00WRD_R_20201420000_01D_MN.rnx
Attempting NAV file download from URL: https://igs.b

In [2]:
%%writefile C:\Users\gauth\Documents\SC4000_proj\ppk_phone_0510.conf
# ppk_phone_0510.conf - config file for RTKLIB PPK solution
#2.9549
pos1-posmode       =kinematic  #
pos1-frequency     =l1+l2+l5   #
pos1-soltype       =combined-nophasereset #
pos1-elmask        =15         #
pos1-snrmask_r     =on         #
pos1-snrmask_b     =on         #
pos1-snrmask_L1    =24,28,28,32,28,28,24,24,24
pos1-snrmask_L2    =34,34,34,34,34,34,34,34,34
pos1-snrmask_L5    =24,20,20,24,20,28,28,20,20 #pos1-snrmask_L5    =24,20,24,24,24,28,28,20,20
pos1-dynamics      =on         #
pos1-tidecorr      =on        #
pos1-ionoopt       =brdc       #
pos1-tropopt       =saas       #
pos1-sateph        =brdc       #
pos1-posopt1       =off        #
pos1-posopt2       =off        #
pos1-posopt3       =off        # 
pos1-posopt4       =off        # 
pos1-posopt5       =off        #
pos1-posopt6       =off        #
pos1-exclsats      =           # 
pos1-navsys        =13         # 
pos2-armode        =off        # 
pos2-gloarmode     =off        # 
pos2-bdsarmode     =on         # 
pos2-arfilter      =on         #
pos2-arthres       =3
pos2-arthresmin    =1.5
pos2-arthresmax    =10

pos2-arthres1      =0.25
pos2-arthres2      =0
pos2-arthres3      =1e-09
pos2-arthres4      =1e-05
pos2-varholdamb    =0.7        # (cyc^2)
pos2-gainholdamb   =0.01

pos2-arlockcnt     =0
pos2-minfixsats    =4
pos2-minholdsats   =5
pos2-mindropsats   =10
pos2-arelmask      =0             # (deg)
pos2-arminfix      =50
pos2-armaxiter     =1
pos2-elmaskhold    =15         # (deg)
pos2-aroutcnt      =4
pos2-maxage        =30         # (s)
pos2-syncsol       =off        #
pos2-slipthres     =0.1        #
pos2-dopthres      =5          #
pos2-rejionno      =1          # (m)
pos2-rejgdop       =30
pos2-niter         =1
pos2-baselen       =0          # (m)
pos2-basesig       =0          # (m)
out-solformat      =llh        # 
out-outhead        =on         #
out-outopt         =on         # 
out-outvel         =off        # 
out-timesys        =gpst       # 
out-timeform       =tow        # 
out-timendec       =3
out-degform        =deg        # 
out-fieldsep       =
out-outsingle      =off        # 
out-maxsolstd      =0          # (m)
out-height         =ellipsoidal # 
out-geoid          =internal   # 
out-solstatic      =all        # 
out-nmeaintv1      =0          # (s)
out-nmeaintv2      =0          # (s)
out-outstat        =residual   # 
stats-eratio1      =400        #
stats-eratio2      =300
stats-eratio5      =100
stats-errphase     =0.006      # (m) 
stats-errphaseel   =0.003      # (m)
stats-errphasebl   =0          # (m/10km)
stats-errdoppler   =1          # (Hz)
stats-snrmax       =50         # (dB.Hz)
stats-errsnr       =0          # (m)
stats-errrcv       =0          # ( )
stats-stdbias      =30         # (m)
stats-stdiono      =0.03       # (m)
stats-stdtrop      =0.3        # (m)

stats-prnaccelh    =1          # (m/s^2)
stats-prnaccelv    =0.1          # (m/s^2)

stats-prnbias      =0.032       # 

stats-prniono      =0.001      # (m)
stats-prntrop      =0.0001     # (m)
stats-prnpos       =0          # (m)
stats-clkstab      =5e-12      # (s/s)
ant1-postype       =llh        # 
ant1-pos1          =0          # 
ant1-pos2          =0          
ant1-pos3          =0          
ant1-anttype       =
ant1-antdele       =0          
ant1-antdeln       =0          
ant1-antdelu       =0          
ant2-postype       =posfile    
ant2-pos1          =0          
ant2-pos2          =0          
ant2-pos3          =0          
ant2-anttype       =
ant2-antdele       =0          
ant2-antdeln       =0          
ant2-antdelu       =0          
ant2-maxaveep      =1
ant2-initrst       =off         
misc-timeinterp    =off         
misc-sbasatsel     =0          
misc-rnxopt1       =
misc-rnxopt2       =
misc-pppopt        =
misc-svrcycle      =5         # (ms)
misc-timeout       =10000      # (ms)
misc-reconnect     =10000      # (ms)
misc-nmeacycle     =5000       # (ms)
misc-buffsize      =32768      # (bytes)
misc-navmsgsel     =all        # (0:all,1:rover,2:base,3:corr)
misc-proxyaddr     =
misc-fswapmargin   =30         # (s)
file-satantfile    =
file-rcvantfile    =
file-staposfile    =C:\Users\gauth\Documents\SC4000_proj\bases.sta
file-geoidfile     =
file-ionofile      =
file-dcbfile       =
file-eopfile       =
file-blqfile       =
file-tempdir       =
file-geexefile     =
file-solstatfile   =
file-tracefile     =

Overwriting C:\Users\gauth\Documents\SC4000_proj\ppk_phone_0510.conf


In [33]:
"""
run_ppk_multi.py - convert raw android files to rinex and run PPK solutions for GDSC_2022
data set with RTKLIB and/or rtklib-py.   MODIFIED FOR TRAIN DATA
"""

import sys
import numpy as np
from os.path import join, isdir, isfile
from glob import glob
from multiprocessing import Pool
import subprocess
from time import time
import os, shutil

# --- PATH MODIFICATIONS TO MATCH YOUR ENVIRONMENT ---
# Append rtklib-py (for definition imports, though Py is disabled)
sys.path.append(r'C:\Users\gauth\Documents\SC4000_proj\rtklib-py')
# Append android_rinex source
sys.path.append(r'C:\Users\gauth\Documents\SC4000_proj\android_rinex\src')
# --------------------------------------------------

# Deferred imports that rely on sys.path being set correctly
try:
    import gnsslogger_to_rnx as rnx
except ImportError:
    print("FATAL ERROR: Could not import 'gnsslogger_to_rnx'. Please ensure the 'android_rinex\\src' folder is at the correct path specified above.")
    sys.exit(1)


# set run parameters
maxepoch = None # max number of epochs, used for debug, None = no limit

# Set solution choices
ENABLE_PY = False        # Use RTKLIB-PY to generate solutions 
ENABLE_RTKLIB = True     # Use RTKLIB to generate solutions
OVERWRITE_RINEX = True  # overwrite existing rinex files
OVERWRITE_SOL = True    # overwrite existing solution files
CLEAN_OLD_FILES = True  # Delete old generated files before regenerating

# --- DATA PATH ---
datadir = r'C:\Users\gauth\Documents\SC4000_proj\train' # Adjusted data directory
# The basefiles search must find the .20o files (RINEX v2 observation file for base)
# Example: slac1500.20o
basefiles = '../*0.2*o' 
# The navfiles search must find the BRDC file (RINEX v3 navigation file)
# Example: BRDC00WRD_R_20201500000_01D_MN.rnx
navfiles = '../*MN.rnx' 

# Setup for RTKLIB 
binpath_rtklib  = r'C:\Users\gauth\Documents\RTKLIB\rnx2rtkp.exe' # Adjusted executable path
cfgfile_rtklib = r'C:\Users\gauth\Documents\SC4000_proj\ppk_phone_0510.conf' # Adjusted config path
soltag_rtklib = '_rtklib' # postfix for solution file names

# Setup for rtklib-py (disabled)
cfgfile = r'D:\kaggle\GSDC2022\config\ppk_phone_0510.py' # This placeholder path doesn't matter since ENABLE_PY=False
soltag_py = '_py0510'  

PHONES = ['GooglePixel4', 'GooglePixel4XL', 'Pixel4Modded', 'GooglePixel5', 'GooglePixel6Pro', 'XiaomiMi8', 'SamsungGalaxyS20Ultra']

# Base stations we're actually using - WGS84 XYZ coordinates
BASE_POS = {
    'slac': [-2703115.9184, -4291767.2037, 3854247.9027],
    'vdcy': [-2497836.5139, -4654543.2609, 3563028.9379],
    'p222': [-2689640.2891, -4290437.3671, 3865050.9313],
}

# input structure for rinex conversion
class args:
    def __init__(self):
        # Input parameters for conversion to rinex
        self.slip_mask = 0 
        self.fix_bias = True
        self.timeadj = 1e-7
        self.pseudorange_bias = 0
        self.filter_mode = 'sync'
        # Optional hader values for rinex files
        self.marker_name = ''
        self.observer = ''
        self.agency = ''
        self.receiver_number = ''
        self.receiver_type = ''
        self.receiver_version = ''
        self.antenna_number = ''
        self.antenna_type = ''

# Copy and read config file (Disabled by ENABLE_PY=False)
if ENABLE_PY:
    import rinex as rn
    import rtkcmn as gn
    from rtkpos import rtkinit
    from postpos import procpos, savesol
    shutil.copyfile(cfgfile, '__ppk_config.py')
    import __ppk_config as cfg

# NEW FUNCTION: Delete old generated files
def clean_old_files(folder):
    """Delete old RINEX and solution files from folder"""
    os.chdir(folder)
    
    # Patterns for files to delete
    patterns_to_delete = [
        join('supplemental', '*.obs'),         # RINEX observation files
        join('supplemental', '*_rtklib.pos'),  # RTKLIB solution files
        join('supplemental', '*_py*.pos'),     # rtklib-py solution files
        join('supplemental', '*.trc'),         # RTKLIB trace files
        join('supplemental', '*.stat'),        # RTKLIB statistics files (if any)
    ]
    
    deleted_count = 0
    for pattern in patterns_to_delete:
        files = glob(pattern)
        for file in files:
            try:
                os.remove(file)
                deleted_count += 1
            except Exception as e:
                print(f"  Could not delete {file}: {e}")
    
    if deleted_count > 0:
        print(f"  Deleted {deleted_count} old file(s)")
    
    return deleted_count

# function to convert single rinex file
def convert_rnx(folder, rawFile, rovFile, slipMask):
    os.chdir(folder)
    argsIn = args()
    argsIn.input_log = rawFile
    argsIn.output = os.path.basename(rovFile)
    argsIn.slip_mask = slipMask
    rnx.convert2rnx(argsIn)

# function to run single RTKLIB-Py solution (disabled)
def run_ppk(folder, rovfile, basefile, navfile, solfile):
    return rovfile

# function to run single RTKLIB solution
def run_rtklib(folder, rovfile, basefile, navfile, solfile, progress_str=""):
    """Run RTKLIB PPK solution with progress tracking"""
    
    # Extract base station name from basefile
    base_station = os.path.basename(basefile).split('0')[0]
    
    # Get dataset/phone info from folder path
    path_parts = folder.split(os.sep)
    dataset_phone = "/".join(path_parts[-2:])  # e.g., "2020-05-15-US-MTV-1/GooglePixel4"
    
    print(f"\n{progress_str}")
    print(f"  Processing: {dataset_phone}")
    print(f"  Base Station: {base_station}")
    print(f"  Rover: {os.path.basename(rovfile)}")
    print(f"  Nav: {os.path.basename(navfile)}")
    
    # create command to run solution
    rtkcmd='%s -x 0 -y 2 -k %s -o %s %s %s %s' % \
        (binpath_rtklib, cfgfile_rtklib, solfile, rovfile, basefile, navfile)
    
    os.chdir(folder)
    
    try:
        result = subprocess.run(rtkcmd, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, 
                               check=True, timeout=120)
        
        # Check if .pos file was actually created
        if isfile(solfile):
            # Get file size for verification
            file_size = os.path.getsize(solfile)
            print(f"  ✓ SUCCESS: Generated {os.path.basename(solfile)} ({file_size} bytes)")
            return True
        else:
            print(f"  ✗ FAILED: .pos file not created")
            return False
            
    except subprocess.TimeoutExpired:
        print(f"  ✗ FAILED: Processing timeout (>120s)")
        return False
    except subprocess.CalledProcessError as e:
        print(f"  ✗ FAILED: RTKLIB error (exit code {e.returncode})")
        return False
    except Exception as e:
        print(f"  ✗ FAILED: {type(e).__name__}: {e}")
        return False

####### Start of main code ##########################

def main():

    # get list of data sets in data path
    datasets = np.sort(os.listdir(datadir))

    # loop through data set folders
    rinexIn = []
    ppkIn = []
    rtklibIn = []
    for dataset in datasets:
        for phone in PHONES:
            # skip if no folder for this phone
            folder = join(datadir, dataset, phone)
            if not isdir(folder):  
                continue
            
            # Change directory is important for glob() calls later
            try:
                os.chdir(folder)
            except FileNotFoundError:
                print(f"Directory not found: {folder}. Skipping.")
                continue

            # --- CLEAN OLD FILES IF REQUESTED ---
            if CLEAN_OLD_FILES:
                clean_old_files(folder)
            # ------------------------------------

            rawFile = join('supplemental', 'gnss_log.txt')
            rovFile = join('supplemental', 'gnss_log.obs')

            rinex = False
            # check if need rinex conversion
            if OVERWRITE_RINEX or not isfile(rovFile):
                # generate list of input parameters for each rinex conversion
                if phone == 'SamsungS20Ultra': 
                    slipMask = 0 
                else:
                    slipMask = 0 
                rinexIn.append((folder, rawFile, rovFile, slipMask))
                print(f"{dataset}/{phone}: Converting RINEX") 
                rinex = True
            
            # check if need to create PPK solution
            try:
                # glob looks in the parent directory (..) for the base/nav files
                baseFile = glob(basefiles)[0] 
                navFile = glob(navfiles)[0]   
                solFile = rovFile[:-4] + soltag_py + '.pos'
                solFile_rtklib = rovFile[:-4] + soltag_rtklib + '.pos'
            except IndexError:
                # This error means glob didn't find the necessary base or nav files in the parent directory
                print(f"{dataset}/{phone}: Missing base or nav file")
                continue
            except Exception as e:
                print(f'{dataset}/{phone}: Error - {e}')
                continue
            
            # Check if RTKLIB solution needs to run
            if ENABLE_RTKLIB and (OVERWRITE_SOL == True or 
                        len(glob(solFile_rtklib)) == 0 or rinex == True):
                # generate list of input/output files for each rtklib ppk solution
                print(f"{dataset}/{phone}: Queued for RTKLIB processing")
                rtklibIn.append((folder, rovFile, baseFile, navFile, solFile_rtklib))

    if len(rinexIn) > 0:
        print(f'\n{"="*60}')
        print(f'Converting {len(rinexIn)} RINEX files...')
        print(f'{"="*60}')
        # Run rinex conversion sequentially
        for idx, input in enumerate(rinexIn, 1):
            print(f"\n[{idx}/{len(rinexIn)}] Converting RINEX...")
            convert_rnx(input[0],input[1],input[2],input[3])
            print(f"  ✓ Conversion complete")

    if ENABLE_RTKLIB and len(rtklibIn) > 0:
        print(f'\n{"="*60}')
        print(f'Calculating {len(rtklibIn)} RTKLIB solutions...')
        print(f'{"="*60}')
        
        success_count = 0
        fail_count = 0
        
        # Run RTKLIB solutions sequentially
        for idx, input in enumerate(rtklibIn, 1):
            progress_str = f"[{idx}/{len(rtklibIn)}] Generating .pos file..."
            success = run_rtklib(input[0], input[1], input[2], input[3], input[4], progress_str)
            
            if success:
                success_count += 1
            else:
                fail_count += 1
        
        print(f'\n{"="*60}')
        print(f'RTKLIB Processing Summary:')
        print(f'  ✓ Successful: {success_count}')
        print(f'  ✗ Failed: {fail_count}')
        print(f'  Total: {len(rtklibIn)}')
        print(f'{"="*60}')

    print(f'\n{"="*60}')
    print('PROCESSING COMPLETE')
    print(f'{"="*60}')

if __name__ == '__main__':
    t0 = time()
    main()
    print(f'\nTotal Runtime: {time() - t0:.1f} seconds')

  Deleted 4 old file(s)
2020-05-15-US-MTV-1/GooglePixel4XL: Converting RINEX
2020-05-15-US-MTV-1/GooglePixel4XL: Queued for RTKLIB processing
  Deleted 4 old file(s)
2020-05-21-US-MTV-1/GooglePixel4: Converting RINEX
2020-05-21-US-MTV-1/GooglePixel4: Queued for RTKLIB processing
  Deleted 4 old file(s)
2020-05-21-US-MTV-1/GooglePixel4XL: Converting RINEX
2020-05-21-US-MTV-1/GooglePixel4XL: Queued for RTKLIB processing
  Deleted 4 old file(s)
2020-05-21-US-MTV-2/GooglePixel4: Converting RINEX
2020-05-21-US-MTV-2/GooglePixel4: Queued for RTKLIB processing
  Deleted 4 old file(s)
2020-05-21-US-MTV-2/GooglePixel4XL: Converting RINEX
2020-05-21-US-MTV-2/GooglePixel4XL: Queued for RTKLIB processing
  Deleted 4 old file(s)
2020-05-28-US-MTV-2/GooglePixel4: Converting RINEX
2020-05-28-US-MTV-2/GooglePixel4: Queued for RTKLIB processing
  Deleted 4 old file(s)
2020-05-28-US-MTV-2/GooglePixel4XL: Converting RINEX
2020-05-28-US-MTV-2/GooglePixel4XL: Queued for RTKLIB processing
  Deleted 4 old fi

In [34]:
"""
diagnose_failed_pos_files.py - Diagnose why certain .pos files failed to load
"""

import os
from os.path import join, isfile
from glob import glob

datadir = r'C:\Users\gauth\Documents\SC4000_proj\train'

# Files that failed according to your output
failed_files = [
    ('2020-06-24-US-MTV-1', 'GooglePixel4'),
    ('2020-06-24-US-MTV-1', 'GooglePixel4XL'),
    ('2020-06-24-US-MTV-2', 'GooglePixel4'),
    ('2020-06-24-US-MTV-2', 'GooglePixel4XL'),
    ('2020-08-03-US-MTV-2', 'GooglePixel4'),
    ('2020-08-03-US-MTV-2', 'GooglePixel4XL'),
    ('2020-08-03-US-MTV-2', 'GooglePixel5'),
    ('2020-11-23-US-MTV-1', 'XiaomiMi8'),
]

print("="*60)
print("DIAGNOSING FAILED .POS FILES")
print("="*60)

for dataset, phone in failed_files:
    print(f"\n{'='*60}")
    print(f"Dataset: {dataset}/{phone}")
    print(f"{'='*60}")
    
    folder = join(datadir, dataset, phone)
    
    # Check if folder exists
    if not os.path.exists(folder):
        print("  ✗ Folder doesn't exist!")
        continue
    
    # Check for .pos file
    pos_file = join(folder, 'supplemental', 'gnss_log_rtklib.pos')
    
    if not isfile(pos_file):
        print("  ✗ .pos file NOT FOUND")
    else:
        # Check file size
        file_size = os.path.getsize(pos_file)
        print(f"  ✓ .pos file exists: {file_size} bytes")
        
        if file_size <= 1020:  # Header-only (approx 1017 bytes)
            print("  ⚠️  HEADER ONLY - No position solutions computed!")
            print("      Reason: RTKLIB failed to compute any positions")
            
            # Check what's in the header
            with open(pos_file, 'r') as f:
                lines = f.readlines()
                # Look for ref pos line
                for line in lines:
                    if 'ref pos' in line:
                        print(f"      {line.strip()}")
        else:
            print(f"  ℹ️  File has data ({file_size} bytes)")
            # Count actual data lines
            with open(pos_file, 'r') as f:
                lines = f.readlines()
                data_lines = [l for l in lines if not l.startswith('%') and l.strip()]
                print(f"      Data lines: {len(data_lines)}")
    
    # Check for base station file
    os.chdir(join(datadir, dataset))
    base_files = glob('*0.2*o')
    if base_files:
        print(f"  ✓ Base file: {base_files[0]}")
    else:
        print("  ✗ Base station file MISSING")
    
    # Check for nav file
    nav_files = glob('*.rnx')
    if nav_files:
        print(f"  ✓ Nav file: {nav_files[0]}")
    else:
        print("  ✗ Navigation file MISSING")
    
    # Check for rover obs file
    rover_file = join(folder, 'supplemental', 'gnss_log.obs')
    if isfile(rover_file):
        rover_size = os.path.getsize(rover_file)
        print(f"  ✓ Rover file: {rover_size} bytes")
    else:
        print("  ✗ Rover observation file MISSING")

print("\n" + "="*60)
print("DIAGNOSIS COMPLETE")
print("="*60)

DIAGNOSING FAILED .POS FILES

Dataset: 2020-06-24-US-MTV-1/GooglePixel4
  ✓ .pos file exists: 1017 bytes
  ⚠️  HEADER ONLY - No position solutions computed!
      Reason: RTKLIB failed to compute any positions
      % ref pos   :  37.416520040 -122.204267280    63.6780
  ✓ Base file: slac1760.20o
  ✓ Nav file: BRDC00WRD_R_20201760000_01D_MN.rnx
  ✓ Rover file: 3989235 bytes

Dataset: 2020-06-24-US-MTV-1/GooglePixel4XL
  ✓ .pos file exists: 1017 bytes
  ⚠️  HEADER ONLY - No position solutions computed!
      Reason: RTKLIB failed to compute any positions
      % ref pos   :  37.416520040 -122.204267280    63.6780
  ✓ Base file: slac1760.20o
  ✓ Nav file: BRDC00WRD_R_20201760000_01D_MN.rnx
  ✓ Rover file: 3958473 bytes

Dataset: 2020-06-24-US-MTV-2/GooglePixel4
  ✓ .pos file exists: 1017 bytes
  ⚠️  HEADER ONLY - No position solutions computed!
      Reason: RTKLIB failed to compute any positions
      % ref pos   :  37.416520040 -122.204267280    63.6780
  ✓ Base file: slac1760.20o
  ✓ 

In [35]:
"""
check_time_overlap.py - Check if rover and base station times overlap
"""

import os
from os.path import join
import numpy as np

datadir = r'C:\Users\gauth\Documents\SC4000_proj\train'

failed_datasets = [
    '2020-06-24-US-MTV-1',
    '2020-08-03-US-MTV-2',
    '2020-11-23-US-MTV-1',
]

def parse_rinex_time_range(filepath):
    """Parse RINEX file to get time range"""
    try:
        with open(filepath, 'r') as f:
            lines = f.readlines()
        
        # Find END OF HEADER
        header_end = 0
        for i, line in enumerate(lines):
            if 'END OF HEADER' in line:
                header_end = i + 1
                break
        
        if header_end == 0:
            return None, None
        
        # Get first and last observation epochs
        first_time = None
        last_time = None
        
        for line in lines[header_end:]:
            # RINEX observation epoch line starts with year
            if len(line) > 26 and line[0] == ' ' and line[1:3].strip().isdigit():
                # Parse time: YY MM DD HH MM SS
                try:
                    year = int('20' + line[1:3].strip())
                    month = int(line[4:6])
                    day = int(line[7:9])
                    hour = int(line[10:12])
                    minute = int(line[13:15])
                    second = float(line[16:26])
                    
                    time_str = f"{year}-{month:02d}-{day:02d} {hour:02d}:{minute:02d}:{second:06.3f}"
                    
                    if first_time is None:
                        first_time = time_str
                    last_time = time_str
                except:
                    continue
        
        return first_time, last_time
    except Exception as e:
        print(f"    Error reading file: {e}")
        return None, None

for dataset in failed_datasets:
    print(f"\n{'='*60}")
    print(f"Checking: {dataset}")
    print(f"{'='*60}")
    
    # Check base station file
    os.chdir(join(datadir, dataset))
    base_files = [f for f in os.listdir() if f.endswith('.20o')]
    
    if base_files:
        base_file = base_files[0]
        print(f"\n📡 Base Station: {base_file}")
        first, last = parse_rinex_time_range(base_file)
        if first and last:
            print(f"    First epoch: {first}")
            print(f"    Last epoch:  {last}")
        else:
            print(f"    ⚠️  Could not parse time range")
    
    # Check rover files for one phone
    phone = 'GooglePixel4'
    rover_file = join(datadir, dataset, phone, 'supplemental', 'gnss_log.obs')
    
    if os.path.exists(rover_file):
        print(f"\n📱 Rover: {phone}")
        first, last = parse_rinex_time_range(rover_file)
        if first and last:
            print(f"    First epoch: {first}")
            print(f"    Last epoch:  {last}")
        else:
            print(f"    ⚠️  Could not parse time range")

print("\n" + "="*60)
print("TIME RANGE CHECK COMPLETE")
print("="*60)


Checking: 2020-06-24-US-MTV-1

📡 Base Station: slac1760.20o
    First epoch: 2020-06-24 00:00:00.000
    Last epoch:  2020-06-24 23:59:30.000

📱 Rover: GooglePixel4
    ⚠️  Could not parse time range

Checking: 2020-08-03-US-MTV-2

📡 Base Station: slac2160.20o
    First epoch: 2020-08-03 00:00:00.000
    Last epoch:  2020-08-03 23:59:30.000

📱 Rover: GooglePixel4
    ⚠️  Could not parse time range

Checking: 2020-11-23-US-MTV-1

📡 Base Station: slac3280.20o
    First epoch: 2020-11-23 00:00:00.000
    Last epoch:  2020-11-23 23:59:30.000

TIME RANGE CHECK COMPLETE


In [36]:
"""
inspect_rover_obs_files.py - Inspect the structure of rover observation files
"""

import os
from os.path import join

datadir = r'C:\Users\gauth\Documents\SC4000_proj\train'

failed_files = [
    ('2020-06-24-US-MTV-1', 'GooglePixel4'),
    ('2020-08-03-US-MTV-2', 'GooglePixel4'),
    ('2020-11-23-US-MTV-1', 'XiaomiMi8'),
]

for dataset, phone in failed_files:
    print(f"\n{'='*60}")
    print(f"Dataset: {dataset}/{phone}")
    print(f"{'='*60}")
    
    rover_file = join(datadir, dataset, phone, 'supplemental', 'gnss_log.obs')
    
    if os.path.exists(rover_file):
        file_size = os.path.getsize(rover_file)
        print(f"File size: {file_size} bytes")
        
        # Read first 50 lines to check format
        with open(rover_file, 'r', errors='ignore') as f:
            lines = []
            for i, line in enumerate(f):
                lines.append(line)
                if i >= 50:
                    break
        
        print("\nFirst 30 lines of file:")
        print("-" * 60)
        for i, line in enumerate(lines[:30]):
            print(f"{i+1:3d}: {line.rstrip()}")
        
        # Check RINEX version
        if lines:
            first_line = lines[0]
            if 'RINEX VERSION' in first_line:
                version = first_line[:20].strip()
                print(f"\n✓ RINEX Version: {version}")
                
                # Check if it's RINEX 3
                if first_line.strip().startswith('3'):
                    print("⚠️  THIS IS RINEX VERSION 3!")
                    print("    RTKLIB may need special configuration for RINEX 3")
            
        # Check for END OF HEADER
        header_end_line = None
        for i, line in enumerate(lines):
            if 'END OF HEADER' in line:
                header_end_line = i
                print(f"\n✓ Header ends at line {i+1}")
                break
        
        if header_end_line is None:
            print("\n✗ NO 'END OF HEADER' FOUND!")
            print("    File may be corrupted or incomplete")
        else:
            # Show first few observation lines
            print("\nFirst 5 observation lines after header:")
            print("-" * 60)
            for i in range(header_end_line + 1, min(header_end_line + 6, len(lines))):
                print(f"{lines[i].rstrip()}")
        
        # Check for time system
        for line in lines[:30]:
            if 'TIME OF FIRST OBS' in line:
                print(f"\n✓ Time of first obs: {line[:60].strip()}")
            if 'TIME SYSTEM' in line:
                print(f"✓ Time system: {line[:60].strip()}")
    else:
        print("✗ Rover file not found!")

print("\n" + "="*60)
print("INSPECTION COMPLETE")
print("="*60)


Dataset: 2020-06-24-US-MTV-1/GooglePixel4
File size: 3989235 bytes

First 30 lines of file:
------------------------------------------------------------
  1:      3.03           O                   M                   RINEX VERSION / TYPE
  2: Android_Rinex                           2025-11-12 18:47:26 PGM / RUN BY / DATE
  3:                                                             MARKER NAME
  4: SMARTPHONE                                                  MARKER TYPE
  5:                                                             OBSERVER / AGENCY
  6:                                                             REC # / TYPE / VERS
  7:                                                             ANT # / TYPE
  8:         0.0000        0.0000        0.0000                  APPROX POSITION XYZ
  9:         0.0000        0.0000        0.0000                  ANTENNA: DELTA H/E/N
 10: G    8 C1C L1C D1C S1C C5Q L5Q D5Q S5Q                      SYS / # / OBS TYPES
 11: R    4 C1C L1C

In [38]:
"""
download_next_day_files_fixed.py - Properly decompress AND convert base station files
"""

import os
from datetime import datetime, timedelta
import requests
import gzip
from os.path import join

datadir = r'C:\Users\gauth\Documents\SC4000_proj\train'
obs_url_base = 'https://geodesy.noaa.gov/corsdata/rinex'
nav_url_base = 'https://igs.bkg.bund.de/root_ftp/IGS/BRDC'
crx2rnx_bin = r"C:\Users\gauth\Documents\RTKLIB\crx2rnx.exe"

# Datasets that need NEXT DAY files
problem_datasets = [
    ('2020-06-24-US-MTV-1', 'slac', '2020', '06', '24'),
    ('2020-06-24-US-MTV-2', 'slac', '2020', '06', '24'),
    ('2020-08-03-US-MTV-2', 'slac', '2020', '08', '03'),
    ('2020-11-23-US-MTV-1', 'slac', '2020', '11', '23'),
]

print("="*60)
print("Converting downloaded base station files...")
print("="*60)

for dataset, station, year, month, day in problem_datasets:
    print(f"\n{'='*60}")
    print(f"Dataset: {dataset}")
    print(f"{'='*60}")
    
    # Calculate next day
    date_obj = datetime(int(year), int(month), int(day))
    next_day = date_obj + timedelta(days=1)
    
    next_year = str(next_day.year)
    next_doy = next_day.timetuple().tm_yday
    next_doy_str = str(next_doy).zfill(3)
    next_year_short = next_year[2:4]
    
    dataset_path = join(datadir, dataset)
    os.chdir(dataset_path)
    
    # ========================================
    # Convert BASE STATION file
    # ========================================
    print(f"\n📡 Converting Base Station File:")
    
    base_fname_gz = f"{station}{next_doy_str}0.{next_year_short}d.gz"
    base_fname_d = f"{station}{next_doy_str}0.{next_year_short}d"
    base_fname_o = f"{station}{next_doy_str}0.{next_year_short}o"
    
    # Check if already converted
    if os.path.exists(base_fname_o):
        print(f"  ✓ Already converted: {base_fname_o}")
        continue
    
    # Check if .gz file exists
    if not os.path.exists(base_fname_gz):
        print(f"  ✗ .gz file not found: {base_fname_gz}")
        continue
    
    print(f"  Step 1: Decompressing {base_fname_gz}...")
    try:
        # Decompress .gz to .d
        with gzip.open(base_fname_gz, 'rb') as f_in:
            with open(base_fname_d, 'wb') as f_out:
                f_out.write(f_in.read())
        
        file_size = os.path.getsize(base_fname_d)
        print(f"    ✓ Decompressed: {base_fname_d} ({file_size} bytes)")
        
    except Exception as e:
        print(f"    ✗ Decompression failed: {e}")
        continue
    
    print(f"  Step 2: Converting {base_fname_d} to RINEX...")
    try:
        # Convert Hatanaka compressed (.d) to RINEX (.o)
        result = os.system(f'{crx2rnx_bin} {base_fname_d}')
        
        if result == 0 and os.path.exists(base_fname_o):
            file_size = os.path.getsize(base_fname_o)
            print(f"    ✓ Converted: {base_fname_o} ({file_size} bytes)")
            
            # Clean up intermediate files
            if os.path.exists(base_fname_gz):
                os.remove(base_fname_gz)
                print(f"    ✓ Cleaned up: {base_fname_gz}")
            if os.path.exists(base_fname_d):
                os.remove(base_fname_d)
                print(f"    ✓ Cleaned up: {base_fname_d}")
        else:
            print(f"    ✗ Conversion failed (exit code: {result})")
            
    except Exception as e:
        print(f"    ✗ Conversion error: {e}")

print("\n" + "="*60)
print("Conversion Summary:")
print("="*60)
print("\nBase station files should now be:")
print("  - slac1770.20o (for 2020-06-25)")
print("  - slac2170.20o (for 2020-08-04)")
print("  - slac3290.20o (for 2020-11-24)")
print("\nNavigation files already extracted:")
print("  - BRDC00WRD_R_20201770000_01D_MN.rnx")
print("  - BRDC00WRD_R_20202170000_01D_MN.rnx")
print("  - BRDC00WRD_R_20203290000_01D_MN.rnx")
print("\nNow run the reprocessing script!")

Converting downloaded base station files...

Dataset: 2020-06-24-US-MTV-1

📡 Converting Base Station File:
  Step 1: Decompressing slac1770.20d.gz...
    ✓ Decompressed: slac1770.20d (3708675 bytes)
  Step 2: Converting slac1770.20d to RINEX...
    ✓ Converted: slac1770.20o (12487640 bytes)
    ✓ Cleaned up: slac1770.20d.gz
    ✓ Cleaned up: slac1770.20d

Dataset: 2020-06-24-US-MTV-2

📡 Converting Base Station File:
  Step 1: Decompressing slac1770.20d.gz...
    ✓ Decompressed: slac1770.20d (3708675 bytes)
  Step 2: Converting slac1770.20d to RINEX...
    ✓ Converted: slac1770.20o (12487640 bytes)
    ✓ Cleaned up: slac1770.20d.gz
    ✓ Cleaned up: slac1770.20d

Dataset: 2020-08-03-US-MTV-2

📡 Converting Base Station File:
  Step 1: Decompressing slac2170.20d.gz...
    ✓ Decompressed: slac2170.20d (3602878 bytes)
  Step 2: Converting slac2170.20d to RINEX...
    ✓ Converted: slac2170.20o (12089677 bytes)
    ✓ Cleaned up: slac2170.20d.gz
    ✓ Cleaned up: slac2170.20d

Dataset: 2020-11

In [39]:
"""
reprocess_failed_files_v2.py - Reprocess with correct next-day files
"""

import sys
import os
from os.path import join, isfile
import subprocess
from glob import glob

sys.path.append(r'C:\Users\gauth\Documents\SC4000_proj\rtklib-py')
sys.path.append(r'C:\Users\gauth\Documents\SC4000_proj\android_rinex\src')

datadir = r'C:\Users\gauth\Documents\SC4000_proj\train'
binpath_rtklib = r'C:\Users\gauth\Documents\RTKLIB\rnx2rtkp.exe'
cfgfile_rtklib = r'C:\Users\gauth\Documents\SC4000_proj\ppk_phone_0510.conf'

# Files with correct next-day base and nav
failed_files = [
    ('2020-06-24-US-MTV-1', 'GooglePixel4', 'slac1770.20o', 'BRDC00WRD_R_20201770000_01D_MN.rnx'),
    ('2020-06-24-US-MTV-1', 'GooglePixel4XL', 'slac1770.20o', 'BRDC00WRD_R_20201770000_01D_MN.rnx'),
    ('2020-06-24-US-MTV-2', 'GooglePixel4', 'slac1770.20o', 'BRDC00WRD_R_20201770000_01D_MN.rnx'),
    ('2020-06-24-US-MTV-2', 'GooglePixel4XL', 'slac1770.20o', 'BRDC00WRD_R_20201770000_01D_MN.rnx'),
    ('2020-08-03-US-MTV-2', 'GooglePixel4', 'slac2170.20o', 'BRDC00WRD_R_20202170000_01D_MN.rnx'),
    ('2020-08-03-US-MTV-2', 'GooglePixel4XL', 'slac2170.20o', 'BRDC00WRD_R_20202170000_01D_MN.rnx'),
    ('2020-08-03-US-MTV-2', 'GooglePixel5', 'slac2170.20o', 'BRDC00WRD_R_20202170000_01D_MN.rnx'),
    ('2020-11-23-US-MTV-1', 'XiaomiMi8', 'slac3290.20o', 'BRDC00WRD_R_20203290000_01D_MN.rnx'),
]

print("="*60)
print("Reprocessing 8 failed files with CORRECT next-day files...")
print("="*60)

success_count = 0
fail_count = 0

for idx, (dataset, phone, base_filename, nav_filename) in enumerate(failed_files, 1):
    print(f"\n[{idx}/8] Processing: {dataset}/{phone}")
    
    folder = join(datadir, dataset, phone)
    rover_file = join(folder, 'supplemental', 'gnss_log.obs')
    base_file = join(datadir, dataset, base_filename)
    nav_file = join(datadir, dataset, nav_filename)
    sol_file = join(folder, 'supplemental', 'gnss_log_rtklib.pos')
    
    # Verify all files exist
    print(f"  Checking files...")
    
    if not isfile(rover_file):
        print(f"    ✗ Rover file missing")
        fail_count += 1
        continue
    else:
        print(f"    ✓ Rover: gnss_log.obs")
    
    if not isfile(base_file):
        print(f"    ✗ Base file missing: {base_filename}")
        print(f"      Run download script first!")
        fail_count += 1
        continue
    else:
        print(f"    ✓ Base:  {base_filename}")
    
    if not isfile(nav_file):
        print(f"    ✗ Nav file missing: {nav_filename}")
        print(f"      Run download script first!")
        fail_count += 1
        continue
    else:
        print(f"    ✓ Nav:   {nav_filename}")
    
    # Run RTKLIB
    print(f"  Running RTKLIB...")
    os.chdir(folder)
    rtkcmd = f'{binpath_rtklib} -x 0 -y 2 -k {cfgfile_rtklib} -o {sol_file} {rover_file} {base_file} {nav_file}'
    
    try:
        subprocess.run(rtkcmd, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, 
                      check=True, timeout=120)
        
        if isfile(sol_file):
            file_size = os.path.getsize(sol_file)
            if file_size > 10000:  # More than just header
                print(f"    ✓ SUCCESS: Generated .pos file ({file_size} bytes)")
                
                # Count actual position lines
                with open(sol_file, 'r') as f:
                    lines = [l for l in f if not l.startswith('%') and l.strip()]
                print(f"    ✓ Position epochs: {len(lines)}")
                success_count += 1
            else:
                print(f"    ⚠️  HEADER ONLY: {file_size} bytes")
                print(f"       Still no valid positions computed")
                fail_count += 1
        else:
            print(f"    ✗ FAILED: No output file created")
            fail_count += 1
            
    except subprocess.TimeoutExpired:
        print(f"    ✗ FAILED: Timeout (>120s)")
        fail_count += 1
    except Exception as e:
        print(f"    ✗ FAILED: {e}")
        fail_count += 1

print("\n" + "="*60)
print("FINAL REPROCESSING SUMMARY:")
print("="*60)
print(f"  ✓ Successful: {success_count}/8")
print(f"  ✗ Failed:     {fail_count}/8")
print("="*60)

if success_count == 8:
    print("\n🎉 ALL 8 FILES SUCCESSFULLY REPROCESSED!")
elif success_count > 0:
    print(f"\n✓ {success_count} files recovered!")
    print(f"⚠️  {fail_count} files still failing - may need further investigation")
else:
    print("\n⚠️  All files still failing - check file integrity")

Reprocessing 8 failed files with CORRECT next-day files...

[1/8] Processing: 2020-06-24-US-MTV-1/GooglePixel4
  Checking files...
    ✓ Rover: gnss_log.obs
    ✓ Base:  slac1770.20o
    ✓ Nav:   BRDC00WRD_R_20201770000_01D_MN.rnx
  Running RTKLIB...
    ✓ SUCCESS: Generated .pos file (175145 bytes)
    ✓ Position epochs: 1298

[2/8] Processing: 2020-06-24-US-MTV-1/GooglePixel4XL
  Checking files...
    ✓ Rover: gnss_log.obs
    ✓ Base:  slac1770.20o
    ✓ Nav:   BRDC00WRD_R_20201770000_01D_MN.rnx
  Running RTKLIB...
    ✓ SUCCESS: Generated .pos file (175281 bytes)
    ✓ Position epochs: 1299

[3/8] Processing: 2020-06-24-US-MTV-2/GooglePixel4
  Checking files...
    ✓ Rover: gnss_log.obs
    ✓ Base:  slac1770.20o
    ✓ Nav:   BRDC00WRD_R_20201770000_01D_MN.rnx
  Running RTKLIB...
    ✓ SUCCESS: Generated .pos file (181711 bytes)
    ✓ Position epochs: 1347

[4/8] Processing: 2020-06-24-US-MTV-2/GooglePixel4XL
  Checking files...
    ✓ Rover: gnss_log.obs
    ✓ Base:  slac1770.20o
   

# Part 3

In [ ]:
""" create_baseline_csv_from_pos.py -  Create csv file PPK solution files using timestamps in reference file
    MODIFIED FOR TRAIN DATA
"""

import os
from os.path import join, isfile
import numpy as np
from datetime import date

########### Input parameters ###############################

# --- MODIFICATION 1: Change 'test' to 'train' ---
DATA_SET = 'train' 
SOL_TAG = '_rtklib'
datapath = r'D:\kaggle\GSDC2022\smartphone-decimeter-2022'
hdrlen = 25    # 25 for RTKLIB, 1 for rtklib-py

# Note: You must ensure 'ground_truths_train.csv' is in the datapath
#       before running this script, as the original notebook instructed.

############################################################

GPS_TO_UTC = 315964782  # second


# get timestamps from existing baseline file
os.chdir(datapath)
# --- MODIFICATION 2: Logic switches to 'ground_truths_train.csv' ---
if DATA_SET == 'train':
    baseline_file = 'ground_truths_train.csv'
else: # 'test'
    baseline_file = 'sample_submission.csv'
base_txt = np.genfromtxt(baseline_file, delimiter=',',invalid_raise=False, 
                         skip_header=1, dtype=str)
msecs_base = base_txt[:,1].astype(np.int64)
phones_base = base_txt[:,0]

# open output file
# --- MODIFICATION 3: Output file name reflects 'train' ---
fout =open('baseline_locations_' + DATA_SET + '_' + \"info\" + '.csv','w')
fout.write('tripId,UnixTimeMillis,LatitudeDegrees,LongitudeDegrees\\n')

# get list of data sets in data path
os.chdir(join(datapath, DATA_SET))\
trips = np.sort(os.listdir())

# loop through data set folders
ix_b = 0
for trip in trips:
    if isfile(trip):
        continue
    phones = os.listdir(trip)
    # loop through phone folders
    for phone in phones:
        # check for valid folder and file
        folder = join(trip, phone)
        if isfile(folder):
            continue
        trip_phone = trip + '/' + phone
        print(trip_phone)

        ix_b = np.where(phones_base == trip_phone)[0]
        sol_path = join(folder, 'supplemental', 'gnss_log' + SOL_TAG + '.pos')
        if isfile(sol_path):
            # parse solution file
            fields = np.genfromtxt(sol_path, invalid_raise=False, skip_header=hdrlen)
            if int(fields[0,1]) > int(fields[-1,1]): # invert if backwards solution
                fields = fields[::-1]
            pos = fields[:,2:5]
            qs = fields[:,5].astype(int)
            nss = fields[:,6].astype(int)
            acc = fields[:,7:13]
            msecs = (1000 * (fields[:,0] * 7 * 24 * 3600 + fields[:,1])).astype(np.int64)
            msecs += GPS_TO_UTC * 1000
        else:
            print('File not found: ', sol_path)
            msecs = msecs_base.copy()
            pos = acc = np.zeros((len(msecs), 3))
            qs = nss = np.zeros(len(msecs))
            
           
        # interpolate to baseline timestamps to fill in missing samples
        llhs = []; stds = []
        for j in range(6):
            if j < 3:
                llhs.append(np.interp(msecs_base[ix_b], msecs, pos[:,j]))
                stds.append(np.interp(msecs_base[ix_b], msecs, acc[:,j],
                                     left=1000, right=1000))
            qsi = np.interp(msecs_base[ix_b], msecs, qs)
            nssi = np.interp(msecs_base[ix_b], msecs, nss)
            
            

        # write results to combined file
        for i in range(len(ix_b)):
                fout.write('%s,%d,%.12f,%.12f,%.2f,%.0f,%.0f,%.3f,%.3f,%.3f\\n' % 
                        (trip_phone, msecs_base[ix_b[i]], llhs[0][i], llhs[1][i],
                         llhs[2][i], qsi[i], nssi[i], stds[0][i], stds[1][i], 
                         stds[2][i]))
fout.close()

SyntaxError: invalid syntax (3962336829.py, line 45)